In [2]:
# TensorFlow library for building and training neural networks
import tensorflow as tf
import numpy as np
import random  # Added for Random Search
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam

# Print available GPUs
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Load MNIST handwritten digit dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize pixel values from [0,255] to [0,1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Variables to store best accuracy and best parameters
best_accuracy = 0
best_params = {}

In [4]:
# Hyperparameter search space
learning_rates = [0.01, 0.001, 0.0001]
hidden_units = [64, 128, 256, 512]
batch_sizes = [16, 32, 64, 128]

# Random Search Settings
num_trials = 6  # Number of random combinations to try

In [5]:
# Explicitly specify GPU usage
with tf.device('/GPU:0'):
    for i in range(num_trials):
        # Randomly select hyperparameters
        lr = random.choice(learning_rates)
        units = random.choice(hidden_units)
        batch = random.choice(batch_sizes)
        
        print(f"Trial {i+1}/{num_trials}: Testing LR={lr}, Units={units}, Batch={batch}")

        # Build MLP model
        model = Sequential([
            Flatten(input_shape=(28, 28)),
            Dense(units, activation='relu'),
            Dense(10, activation='softmax')
        ])

        # Compile the model
        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        # Train model (small epochs for search)
        model.fit(
            X_train, y_train,
            epochs=3, 
            batch_size=batch,
            validation_split=0.2,
            verbose=0
        )

        # Evaluate model
        _, acc = model.evaluate(X_test, y_test, verbose=0)
        print(f"-> Trial Accuracy: {acc:.4f}")

        # Update best model if accuracy improves
        if acc > best_accuracy:
            best_accuracy = acc
            best_params = {
                'learning_rate': lr,
                'units': units,
                'batch_size': batch
            }

Trial 1/6: Testing LR=0.01, Units=128, Batch=128
-> Trial Accuracy: 0.9633
Trial 2/6: Testing LR=0.001, Units=64, Batch=32
-> Trial Accuracy: 0.9650
Trial 3/6: Testing LR=0.01, Units=64, Batch=64
-> Trial Accuracy: 0.9592
Trial 4/6: Testing LR=0.0001, Units=512, Batch=128
-> Trial Accuracy: 0.9346
Trial 5/6: Testing LR=0.01, Units=512, Batch=128
-> Trial Accuracy: 0.9697
Trial 6/6: Testing LR=0.001, Units=256, Batch=128
-> Trial Accuracy: 0.9677


In [6]:
# Display best hyperparameters
print("\nBest Accuracy from Random Search:", best_accuracy)
print("Best Hyperparameters:", best_params)

# Build final model using best hyperparameters
final_model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(best_params['units'], activation='relu'),
    Dense(10, activation='softmax')
])

# Compile final model
final_model.compile(
    optimizer=Adam(learning_rate=best_params['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


Best Accuracy from Random Search: 0.9696999788284302
Best Hyperparameters: {'learning_rate': 0.01, 'units': 512, 'batch_size': 128}


In [7]:
# Train final model with optimal hyperparameters
history = final_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    verbose=1
)

# Evaluate final model on unseen test data
loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

# Print final accuracy
print(f"\nFinal Test Accuracy using Best Random Search Hyperparameters: {accuracy*100:.2f}%")

Epoch 1/10
375/375 [==============================] - 2s 4ms/step - loss: 0.2305 - accuracy: 0.9303 - val_loss: 0.1504 - val_accuracy: 0.9557
Epoch 2/10
375/375 [==============================] - 1s 3ms/step - loss: 0.1147 - accuracy: 0.9658 - val_loss: 0.1235 - val_accuracy: 0.9647
Epoch 3/10
375/375 [==============================] - 1s 3ms/step - loss: 0.0896 - accuracy: 0.9734 - val_loss: 0.1470 - val_accuracy: 0.9611
Epoch 4/10
375/375 [==============================] - 1s 3ms/step - loss: 0.0786 - accuracy: 0.9771 - val_loss: 0.1326 - val_accuracy: 0.9704
Epoch 5/10
375/375 [==============================] - 1s 3ms/step - loss: 0.0639 - accuracy: 0.9810 - val_loss: 0.1558 - val_accuracy: 0.9652
Epoch 6/10
375/375 [==============================] - 1s 3ms/step - loss: 0.0648 - accuracy: 0.9812 - val_loss: 0.1839 - val_accuracy: 0.9641
Epoch 7/10
375/375 [==============================] - 1s 3ms/step - loss: 0.0613 - accuracy: 0.9825 - val_loss: 0.2003 - val_accuracy: 0.9644
Epoch 